# Clasificador usando el algortimo de Árbol de Decisión (*Decision Tree*)

## 1. Importar las librerías

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Cargar el Dataset Iris

El conjunto de datos IRIS es probablemente el primer conjunto de datos que los estudiantes y principiantes experimentan mientras aprenden ciencia de datos y aprendizaje automático (*machine learning*). El conjunto de datos IRIS es simple y fácil de entender, y ha sido utilizado por estudiantes, investigadores y profesionales durante décadas para aprender los conceptos básicos de clasificación y análisis de datos. 

El conjunto de datos IRIS fue publicado en 1936 por el estadístico británico Ronald A. Fisher en el artículo “**El uso de mediciones múltiples en problemas taxonómicos**” y, desde entonces, el conjunto de datos IRIS se ha hecho conocido como el “Hola mundo” del aprendizaje automático (ML) porque proporciona una introducción muy efectiva a los conjuntos de datos y algoritmos del mundo real para los estudiantes.

![Descripción](iris.png)

El conjunto de datos Iris es un conjunto de datos de **150 filas** y **4 características** de medidas de flores de iris (**longitud del sépalo, ancho del sépalo, longitud de los pétalos, ancho de los pétalos**) en **tres especies** — **Setosa, Versicolor y Virginica**. La longitud y el ancho de los pétalos son las dos características más correlacionadas con las especies, lo que las convierte en los predictores más fuertes. Es el primer conjunto de datos estándar para aprender la clasificación porque es pequeño, limpio y no requiere limpieza de datos antes del modelado.

Las cuatro características medidas (en centímetros) para cada flor:
Estas son las medidas que utilizamos para hacer nuestra predicción. Puedes pensar en ellos como las piezas de un rompecabezas.
- Longitud del sépalo (cm): El sépalo es la parte de la flor que protege el brote antes de florecer. Esta característica es una medida de la longitud del sépalo. 
- Ancho del sépalo (cm): es una medida del ancho del sépalo.
- Longitud del pétalo (cm): El pétalo es la parte muy colorida de la flor. Esta es una medida de cuánto tiempo dura. 
- Ancho del pétalo (cm): es una medida del ancho del pétalo. 

![Descripción](iris_dimensions.png)

Tomado de: [Guvi](https://www.guvi.in/blog/iris-dataset-explained-features/)

Las clases objetivo (los resultados): 
- Iris Setosa (0): Esta especie se distingue claramente de las otras dos especies de Iris, por lo que puede considerarse como el "modo fácil" de este conjunto de datos para un clasificador. 
- Iris Versicolor (1): Esta especie es muy similar a Virginica, lo que significa que es más difícil distinguirlas. ¡Ahí radica la verdadera dificultad!
- Iris Virginica (2): La tercera especie que es más probable que se confunda con Versicolor basándose únicamente en las medidas de dimensiones. 

In [ ]:
# Cargar el dataset
# iris = load_iris()
# iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)

# iris_df = iris.frame
# iris_df.to_csv('dataset/iris.csv')
iris_df = pd.read_csv('dataset/iris.csv', header=0, index_col=0)

In [ ]:
# Nombres de las características
# iris.feature_names

In [ ]:
iris_df.columns

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Dimensiones del dataset
iris_df.shape

In [ ]:
# Tipos de datos del dataset
iris_df.dtypes

In [ ]:
# Primeros registros del dataset
iris_df.head(5)

In [ ]:
# Últimos registros del dataset
iris_df.tail(5)

In [ ]:
# Descripción del dataset
iris_df.describe()

In [ ]:
# Iris Setosa(0)
iris_df[iris_df.target == 0].describe()

In [ ]:
# Iris Versicolor(1)
iris_df[iris_df.target == 1].describe()

In [ ]:
# Iris Virginica(2)
iris_df[iris_df.target == 2].describe()

In [ ]:
iris_df.hist()

In [ ]:
# Histograma para la clase setosa (0)
iris_df[iris_df.target==0].hist()

In [ ]:
# Histograma para la clase Versicolor (1)
iris_df[iris_df.target==1].hist()

In [ ]:
# Histograma para la clase Virginica (2)
iris_df[iris_df.target==2].hist()

In [ ]:
iris_df['target'].value_counts().plot(kind='bar')
plt.title('Distribución de las especies')
plt.xlabel('Especie (Setosa, Versicolor, Virginica)')
plt.ylabel('Frecuencia')
plt.show()


## Verificar valores faltantes 

In [ ]:
iris_df.isnull().sum()

In [ ]:
print("Porcentaje de valores nulos ")
iris_df.isnull().mean()*100


# Correlación entre variables

In [ ]:
plt.figure(figsize=(10, 8))     # 10 x 8 pulgadas
sns.heatmap(iris_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlación de características')
plt.tight_layout()
plt.show()

**Insights clave**: 
- Fuertemente positivo: (r > 0.8) petal_length <-> petal_width: r = 0.96 multicolineal
- Negativo: (r < -0.3) sepal_width <-> petal_length: r = -0.43

*¿Por qué es importante?* La correlación solo refleja relaciones lineales. Dos variables pueden estar fuertemente relacionadas de forma no lineal y aun así mostrar una correlación cercana a cero. Además, la correlación no implica causalidad. Las características altamente correlacionadas no aportan información nueva. Considere eliminar una. 

## Visualizar las distribuciones

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 10))
axes = axes.flatten()
for i, col in enumerate(iris_df.columns):
    iris_df[col].hist(bins=50, ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Mapea los números de target a los nombres reales de las especies
target_names = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
iris_plot = iris_df.copy()
iris_plot['species'] = iris_plot['target'].map(target_names)

features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

# Crear cuadrícula de 2x2 gráficos
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.histplot(
        data=iris_plot, 
        x=col, 
        hue='species', 
        kde=True,          # Agrega curva de densidad
        ax=axes[i], 
        palette='tab10', 
        element="step",
        alpha=0.6
    )
    axes[i].set_title(f'Histograma: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Medida (cm)')
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución por Variable según Especie (Iris Dataset)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Gráfico de Dispersión con Codificación de colores

In [ ]:
sns.set_theme(style='whitegrid')
sns.scatterplot(data=iris_df,
    x='sepal length (cm)',
    y='petal length (cm)',
    hue='target', palette='deep')
plt.title('Iris Dataset — Sepal vs Petal Length')
plt.show()